In [0]:
from pyspark.sql.functions import current_timestamp, col

base_path = "/Volumes/workspace/freshmart_bronze/raw_files"

# 1. Ingest Raw Sales into Bronze Delta Table
sales_stream = (
    spark.readStream
    .format("cloudFiles")
    .option("cloudFiles.format", "csv")
    .option("header", "true")
    .option("cloudFiles.schemaLocation", f"{base_path}/_schemas/sales")
    .load(f"{base_path}/sales/")
    .withColumn("ingestion_time", current_timestamp())
    .withColumn("source_file", col("_metadata.file_path"))
)

(
    sales_stream.writeStream
    .format("delta")
    .option("checkpointLocation", f"{base_path}/_checkpoints/sales")
    .option("mergeSchema", "true")
    .trigger(availableNow=True)
    .toTable("workspace.freshmart_bronze.raw_sales")
    .awaitTermination()
)

# 2. Ingest Raw Customers into Bronze Delta Table
cust_stream = (
    spark.readStream
    .format("cloudFiles")
    .option("cloudFiles.format", "csv")
    .option("header", "true")
    .option("cloudFiles.schemaLocation", f"{base_path}/_schemas/customers")
    .load(f"{base_path}/customers/")
    .withColumn("ingestion_time", current_timestamp())
    .withColumn("source_file", col("_metadata.file_path"))
)

(
    cust_stream.writeStream
    .format("delta")
    .option("checkpointLocation", f"{base_path}/_checkpoints/customers")
    .option("mergeSchema", "true")
    .trigger(availableNow=True)
    .toTable("workspace.freshmart_bronze.raw_customers")
    .awaitTermination()
)

print("Bronze ingestion complete!")

Bronze ingestion complete!


In [0]:
%sql
SELECT * FROM workspace.freshmart_bronze.raw_sales

order_id,order_date,customer_id,product_id,product_name,category,qty,unit_price,_rescued_data,ingestion_time,source_file
O101,2026-11-01,C001,P01,wireless mouse,Electronics,2,500,null,2026-11-25T17:24:01.062Z,/Volumes/workspace/freshmart_bronze/raw_files/sales/sales_day1.csv
O101,2026-11-01,C001,P01,wireless mouse,Electronics,2,500,null,2026-11-25T17:24:01.062Z,/Volumes/workspace/freshmart_bronze/raw_files/sales/sales_day1.csv
O102,2026-11-01,C002,P02,Coffee Mug,Kitchen,3,250,null,2026-11-25T17:24:01.062Z,/Volumes/workspace/freshmart_bronze/raw_files/sales/sales_day1.csv
O103,2026-11-02,C003,P03,Notebook Pack,Stationery,5,120,null,2026-11-25T17:24:01.062Z,/Volumes/workspace/freshmart_bronze/raw_files/sales/sales_day1.csv
O104,2026-11-02,C001,P02,Coffee Mug,Kitchen,1,-250,null,2026-11-25T17:24:01.062Z,/Volumes/workspace/freshmart_bronze/raw_files/sales/sales_day1.csv
O105,2026-11-03,C002,P04,null,Electronics,1,1500,null,2026-11-25T17:24:01.062Z,/Volumes/workspace/freshmart_bronze/raw_files/sales/sales_day1.csv
O106,2026-11-03,C003,P01,WIRELESS MOUSE,Electronics,1,500,null,2026-11-25T17:24:01.062Z,/Volumes/workspace/freshmart_bronze/raw_files/sales/sales_day1.csv
